## Installing dependencies

RAG Pipeline:
document loading
embedded model
llm model
vector database

In [2]:
import sys
!python -m pip install pypdf sentence-transformers faiss-cpu torch huggingface_hub groq langsmith python-dotenv tiktoken

print('\nInstallation completed')


Installation completed



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
print("Installing Dependencies")

import sys
!python -m pip install langchain langchain-community langchain-text-splitters langchain-huggingface langchain-groq langchain-google-genai langchain-openai langchain-core faiss-cpu pypdf sentence-transformers transformers torch huggingface_hub groq langsmith python-dotenv tiktoken


print('\nInstallation completed')

Installing Dependencies

Installation completed



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
LLM_Provider = 'groq'
LLM_Model = 'openai/gpt-oss-20b'
CORPUS_PATH = "./zyro-dynamics-hr-corpus"

In [5]:
%pip install langchain-community langchain-core langchain-text-splitters langchain-chroma pypdf

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import os
from dotenv import load_dotenv
print("Did it find the .env file?:",load_dotenv())
if os.getenv("GROQ_API_KEY"):
    print("GROQ api key is imported..")
else:
    print("GROQ api key is not imported")
from langchain_community.document_loaders import PyPDFDirectoryLoader 
from langchain_text_splitters import RecursiveCharacterTextSplitter 


C:\Users\ABHINANDH UNNI\AppData\Local\Temp\ipykernel_23600\3242123715.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader


Did it find the .env file?: True
GROQ api key is imported..


c:\Users\ABHINANDH UNNI\OneDrive\Desktop\Intelligent Rag Bot\.venv-1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
loader=PyPDFDirectoryLoader(CORPUS_PATH)
documents=loader.load()
print(f"loaded {len(documents)} documents")

loaded 39 documents


In [8]:
splitter = RecursiveCharacterTextSplitter(
chunk_size=800,
chunk_overlap=100
)

chunks=splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks")

Created 107 chunks


In [9]:
from langchain_huggingface import HuggingFaceEmbeddings

model=HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2'
)
print("Embedded done")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2827.05it/s]


Embedded done


In [10]:
from langchain_community.vectorstores import FAISS 
vectorstore = FAISS.from_documents(chunks,model)

retriever = vectorstore.as_retriever(search_kwargs ={"k":3})
print("Done!..")

Done!..


In [17]:
from langchain_groq import ChatGroq 
llm_model = ChatGroq(
    model=LLM_Model,
    groq_api_key=os.getenv("GROQ_API_KEY"),
    temperature=0.6,
    max_tokens=500
)

print("LLM model groq initialsed")

LLM model groq initialsed


In [41]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langsmith import traceable
RAG_PROMPT= ChatPromptTemplate.from_template(
""" 
You are an HR assistant.Your goal is to accurately answer the employee questions using ONLY the provided context below.
context;{context}
question:{question}
"""
)

def format_docs(docs):
    return ".\n\n".join(d.page_content for d in docs)

@traceable(name="rag_chain")

def rag_chain(question : str):
    docs=retriever.invoke(question)
    context=format_docs(docs)
    chain=RAG_PROMPT | llm_model | StrOutputParser()
    answer=chain.invoke({"context":context,"question":question})
    return {"answer":answer}

In [52]:
rag_chain("What is the code of conduct?")

{'answer': '**Zyro Dynamics Pvt. Ltd. Code of Conduct (Doc Code: ZDL‑H‑004)**  \n\n- **Purpose:** To guide employees, contractors, consultants, and interns in behaving with honesty, integrity, and professionalism, both inside and outside the company.  \n- **Scope:** Applies to all individuals employed by or associated with Zyro Dynamics, regardless of employment type, location, or role. It covers workplace behavior, client sites, official travel, company events, and any digital or social media interactions linked to the company.  \n- **Key Principles:**  \n  1. Treat everyone with dignity and respect, regardless of designation, gender, age, religion, nationality, sexual orientation, or disability.  \n  2. Maintain a professional tone and language in all communications.  \n  3. Resolve disagreements through respectful dialogue; if unresolved, involve the reporting manager or HR.  \n  4. Avoid offensive or discriminatory language, inappropriate content, or any conduct that creates a host

In [31]:
GUARDIAL_PROMPT=ChatPromptTemplate.from_template("""
You are a scope classifier for an HR assistant.Decide whether the question below is something an HR assistant should answer (company leave policy,
reimbursement,code of conduct,or other internal HR topics) or something out of scope (general knowledge,coding help,unrelated small talk,etc).
Respond with exactly one word: IN_SCOPE or OUT_OF_SCOPE.
Question: {question}"""
)
REFUSAL_MESSAGE = (
    "I'm an HR assistant and can only help with questions about company HR "
    "policies (leave,reimbursement,code of conduct,etc.).I don't have "
    "information to answer that question."
)

In [53]:
def ask_bot(question:str):
    guardial_chain=GUARDIAL_PROMPT | llm_model | StrOutputParser()
    verdict= guardial_chain.invoke({"question" : question}).strip().upper()
    if "OUT_OF_SCOPE" in verdict or verdict != "IN_SCOPE":
        return {"answer" : REFUSAL_MESSAGE , "sources" : [ ]}
    return rag_chain(question)

print("Guardial Initialised")

Guardial Initialised


In [55]:
test_question=["How many casual leaves do i get per year?",
"What is the internet reimbursement limit?",
"What is the capital of France?"]

for i,q in enumerate(test_question,1):
    result=ask_bot(q)
    print(f"Q{i}: {q}")
    print(f"A{i}: {result['answer']}")
    sources=result.get("sources")
    if sources:
        print(
            f"Sources: "
            f"{[d.metadata.get('source') for d in result['sources']]}"
        )
    print("-" * 60)


Q1: How many casual leaves do i get per year?
A1: I’m sorry, but the information about the exact number of casual leaves you receive per year isn’t included in the details you provided. If you need that specific figure, please refer to your employee handbook or contact the HR team for clarification.
------------------------------------------------------------
Q2: What is the internet reimbursement limit?
A2: The internet reimbursement limit is **Rs. 1,000 per month** for Full‑Remote employees at level 5 and above.
------------------------------------------------------------
Q3: What is the capital of France?
A3: I'm an HR assistant and can only help with questions about company HR policies (leave,reimbursement,code of conduct,etc.).I don't have information to answer that question.
------------------------------------------------------------


In [ ]:
pip install pandas ipykernel

   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.6 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.6 MB 2.5 MB/s eta 0:00:04
   ----- ---------------------------------- 1.3/9.6 MB 2.4 MB/s eta 0:00:04
   ------- -------------------------------- 1.8/9.6 MB 2.5 MB/s eta 0:00:04
   ---------- ----------------------------- 2.6/9.6 MB 2.8 MB/s eta 0:00:03
   --------------- ------------------------ 3.7/9.6 MB 3.2 MB/s eta 0:00:02
   ------------------ --------------------- 4.5/9.6 MB 3.4 MB/s eta 0:00:02
   ----------------------- ---------------- 5.8/9.6 MB 3.7 MB/s eta 0:00:02
   ---------------------------- ----------- 6.8/9.6 MB 3.9 MB/s eta 0:00:01
   -------------------------------- ------- 7.9/9.6 MB 4.1 MB/s eta 0:00:01
   ---------------------------------- ----- 8.4/9.6 MB 3.9 MB/s eta 0:00:01
   ---------------------------------------  9.4/9.6 MB 4.1 MB/s eta 0:00:01
   -----------------------


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [63]:
import pandas as pd 
test_df=pd.read_csv("test (1).csv")
test_df

,question_id,question
0,Q01,How does my Earned Leave accrue every month?
1,Q02,How much Earned Leave can I carry forward to n...
2,Q03,How many weeks of Maternity Leave am I entitle...
3,Q04,Do I need a medical certificate for sick leave?
4,Q05,What date does my salary get credited every mo...
5,Q06,What's the salary range for an L4 Senior emplo...
6,Q07,What medical insurance coverage does the compa...
7,Q08,When am I put on a Performance Improvement Plan?
8,Q09,What's the timeline for the Annual Performance...
9,Q10,Am I eligible to work from home at my grade?


In [64]:
answers=[]
for i,row in test_df.iterrows():
    q=row["question"]
    result=ask_bot(q)
    answers.append(result["answer"])
    print(f"[{i+1}/{len(test_df)}] {q}\n-> {result['answer']}\n{'-'*60}")



[1/20] How does my Earned Leave accrue every month?
-> Earned Leave (EL) accrues as follows:

| Period | Accrual Rate | Notes |
|--------|--------------|-------|
| **During Probation** | 0.5 days per month | These days are added to your balance each month, but they can only be used once you pass probation. |
| **After 1 year of continuous service** | 1.25 days per month | You become eligible for 15 days of EL after completing one year (with at least 240 days worked that year). From that point onward, you earn 1.25 days of EL every month. |

So, if you’re past probation, you’ll receive 1.25 EL days each month. If you’re still on probation, you’ll receive 0.5 EL days each month until you confirm your probation status.
------------------------------------------------------------
[2/20] How much Earned Leave can I carry forward to next year?
-> Earned Leave is not subject to the “no carry‑forward” rule that applies to Casual and Sick Leave.  
The policy does not set a specific maximum for 

In [66]:
test_df["answer"]=answers
submission=test_df[["question_id","answer"]]
submission.to_csv("submission.csv",index=False)
submission.head()

,question_id,answer
0,Q01,Earned Leave (EL) accrues as follows:\n\n| Per...
1,Q02,Earned Leave is not subject to the “no carry‑f...
2,Q03,**Maternity Leave entitlement**\n\n| Child | P...
3,Q04,Yes. \nIf your sick leave extends beyond **2 ...
4,Q05,Your salary is credited to your registered ban...
